# Phase 2 — Ngày 3–4: Alignment gate và content embedding

Notebook **self-contained**, đọc output của `01_preprocessing.ipynb` và tạo embedding đầu vào cho RQ-VAE.

Luồng xử lý:

1. Đọc `product_versions.parquet` và `global_products.parquet`.
2. Load `jinaai/jina-embeddings-v5-text-nano-clustering` ở chế độ GPU.
3. Encode một mẫu sản phẩm có nhiều metadata versions.
4. Đo cosine giữa các versions cùng `product_id` so với random pairs.
5. Nếu alignment gate đạt, encode toàn bộ metadata versions trong một lượt.
6. Gộp theo công thức `Normalize(Mean(version_embeddings))`.
7. Xuất `global_product_embeddings.f16.npy` và index Parquet cùng thứ tự.

Model được chọn cho mục tiêu clustering, có embedding gốc 768 chiều và hỗ trợ Matryoshka 256 chiều. License của model là CC BY-NC 4.0, phù hợp cho nghiên cứu phi thương mại; cần xem lại giấy phép trước khi dùng production.

## 0. Cấu hình

In [ ]:
from pathlib import Path

PREPROCESSED_ROOT = None
OUTPUT_ROOT = None

MODEL_NAME = "jinaai/jina-embeddings-v5-text-nano-clustering"
EMBEDDING_DIM = 256
BATCH_SIZE = 128

ALIGNMENT_PRODUCT_SAMPLE = 20_000
ALIGNMENT_MEDIAN_MIN = 0.65
ALIGNMENT_MARGIN_MIN = 0.15
STOP_IF_ALIGNMENT_FAILS = True

SEED = 2026
AUTO_INSTALL_DEPENDENCIES = True
USE_ALL_GPUS = True
RESUME_VERSION_ENCODING = True
RESET_OUTPUT = False

print("Configuration loaded.")

## 1. Cài dependency và kiểm tra GPU

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys


def installed_version(distribution):
    try:
        return metadata.version(distribution)
    except metadata.PackageNotFoundError:
        return None


required = {
    "sentence-transformers": "5.2.0",
    "transformers": "5.1.0",
    "peft": "0.15.2",
}

if AUTO_INSTALL_DEPENDENCIES:
    from packaging.version import Version

    missing_or_old = [
        f"{name}>={minimum}"
        for name, minimum in required.items()
        if installed_version(name) is None or Version(installed_version(name)) < Version(minimum)
    ]
    if missing_or_old:
        print("Installing:", missing_or_old)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing_or_old])

import torch
from sentence_transformers import SentenceTransformer

print("torch:", torch.__version__)
print("sentence-transformers:", installed_version("sentence-transformers"))
print("transformers:", installed_version("transformers"))
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {props.name}, {props.total_memory / 2**30:.1f} GiB")

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 02 requires a GPU. Enable a Kaggle accelerator before full encoding.")

## 2. Tìm artifacts từ notebook 01

In [ ]:
import json
import os
import shutil
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def is_preprocessed_root(path):
    path = Path(path)
    return (
        (path / "global_products.parquet").is_file()
        and (path / "product_versions.parquet").is_file()
    )


def locate_preprocessed_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_preprocessed_root(root):
            return root
        raise FileNotFoundError(f"Notebook 01 artifacts were not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        Path("/kaggle/working/preprocessed"),
        cwd / "preprocessed",
        cwd.parent / "preprocessed",
    ]
    for candidate in candidates:
        if is_preprocessed_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for manifest_path in kaggle_input.glob("**/preprocessing_manifest.json"):
            if is_preprocessed_root(manifest_path.parent):
                return manifest_path.parent.resolve()
        for global_path in kaggle_input.glob("**/global_products.parquet"):
            if is_preprocessed_root(global_path.parent):
                return global_path.parent.resolve()
    raise FileNotFoundError(
        "Notebook 01 output was not found. Add it as a Kaggle Dataset or set PREPROCESSED_ROOT."
    )


PREPROCESSED_ROOT = locate_preprocessed_root(PREPROCESSED_ROOT)
if OUTPUT_ROOT is None:
    OUTPUT_ROOT = (
        Path("/kaggle/working/embeddings")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "embeddings"
    )
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser().resolve()
if OUTPUT_ROOT.exists() and RESET_OUTPUT:
    if OUTPUT_ROOT.name != "embeddings":
        raise ValueError(f"Refusing to delete a path without the expected safe name: {OUTPUT_ROOT}")
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

product_versions_path = PREPROCESSED_ROOT / "product_versions.parquet"
product_versions = pd.read_parquet(product_versions_path)
global_products = pd.read_parquet(PREPROCESSED_ROOT / "global_products.parquet")

assert global_products["product_index"].tolist() == list(range(len(global_products)))
assert global_products["product_id"].is_unique
assert tuple(pq.read_schema(product_versions_path).names) == (
    "product_id", "normalized_text"
)

print("PREPROCESSED_ROOT:", PREPROCESSED_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("product metadata versions:", f"{len(product_versions):,}")
print("global products:", f"{len(global_products):,}")
print("multi-version products:", f"{global_products['version_count'].gt(1).sum():,}")

## 3. Load clustering encoder

In [ ]:
model_dtype = torch.float16
model = SentenceTransformer(
    MODEL_NAME,
    trust_remote_code=True,
    device="cuda:0",
    model_kwargs={"dtype": model_dtype},
)

base_dimension = model.get_sentence_embedding_dimension()
max_sequence_length = int(model.max_seq_length)
assert base_dimension >= EMBEDDING_DIM


def encode_texts(texts, pool=None, show_progress=True):
    kwargs = {
        "batch_size": BATCH_SIZE,
        "show_progress_bar": show_progress,
        "convert_to_numpy": True,
        "normalize_embeddings": True,
        "truncate_dim": EMBEDDING_DIM,
    }
    if pool is None:
        kwargs["device"] = "cuda:0"
    else:
        kwargs["pool"] = pool
    embeddings = model.encode(texts, **kwargs)
    embeddings = np.asarray(embeddings, dtype=np.float32)
    if embeddings.ndim != 2 or embeddings.shape[1] != EMBEDDING_DIM:
        raise ValueError(f"Unexpected embedding shape: {embeddings.shape}")
    if not np.isfinite(embeddings).all():
        raise ValueError("Encoder returned NaN or Inf")
    return embeddings


smoke = encode_texts(
    ["title: Wireless headphones", "title: Kabellose Kopfhörer"],
    show_progress=False,
)
print("base dimension:", base_dimension)
print("output dimension:", smoke.shape[1])
print("max sequence length:", max_sequence_length)
print("norms:", np.linalg.norm(smoke, axis=1))
print("cross-language smoke cosine:", float(smoke[0] @ smoke[1]))

## 4. Alignment sample

Không dùng cột locale. Positive pairs được xác định duy nhất bằng việc hai metadata rows có cùng global `product_id`. Random pairs là đối chứng âm.

In [ ]:
multi_version = global_products.loc[
    global_products["version_count"].gt(1), ["product_id", "version_count"]
]
sample_size = min(ALIGNMENT_PRODUCT_SAMPLE, len(multi_version))
sample_ids = set(multi_version.sample(sample_size, random_state=SEED)["product_id"].tolist())

alignment_versions = product_versions.loc[
    product_versions["product_id"].isin(sample_ids),
    ["product_id", "normalized_text"],
].reset_index(drop=True)
observed_counts = alignment_versions["product_id"].value_counts()
expected_counts = multi_version.set_index("product_id").loc[list(sample_ids), "version_count"]
assert all(
    int(observed_counts[product_id]) == int(expected_counts[product_id])
    for product_id in sample_ids
)

print("sample products:", f"{len(sample_ids):,}")
print("sample metadata versions:", f"{len(alignment_versions):,}")
display(alignment_versions.head(3))

## 5. Encode alignment sample và chạy gate

In [ ]:
alignment_embeddings = encode_texts(alignment_versions["normalized_text"].tolist())
alignment_product_ids = alignment_versions["product_id"].to_numpy()

same_product_scores = []
for _, indices in alignment_versions.groupby("product_id", sort=False).indices.items():
    group_embeddings = alignment_embeddings[np.asarray(indices)]
    similarity = group_embeddings @ group_embeddings.T
    upper = np.triu_indices(len(group_embeddings), k=1)
    same_product_scores.extend(similarity[upper].tolist())
same_product_scores = np.asarray(same_product_scores, dtype=np.float32)

rng = np.random.default_rng(SEED)
negative_indices = rng.permutation(len(alignment_embeddings))
same_id = alignment_product_ids == alignment_product_ids[negative_indices]
attempts = 0
while same_id.any() and attempts < 20:
    negative_indices[same_id] = rng.permutation(len(alignment_embeddings))[: int(same_id.sum())]
    same_id = alignment_product_ids == alignment_product_ids[negative_indices]
    attempts += 1
if same_id.any():
    for row_index in np.flatnonzero(same_id):
        candidate = (row_index + 1) % len(alignment_embeddings)
        while alignment_product_ids[candidate] == alignment_product_ids[row_index]:
            candidate = (candidate + 1) % len(alignment_embeddings)
        negative_indices[row_index] = candidate
assert not np.any(alignment_product_ids == alignment_product_ids[negative_indices])
random_pair_scores = np.sum(alignment_embeddings * alignment_embeddings[negative_indices], axis=1)

alignment_metrics = {
    "sample_products": len(sample_ids),
    "sample_versions": len(alignment_versions),
    "same_pair_count": len(same_product_scores),
    "same_cosine_mean": float(np.mean(same_product_scores)),
    "same_cosine_p10": float(np.quantile(same_product_scores, 0.10)),
    "same_cosine_median": float(np.median(same_product_scores)),
    "same_cosine_p90": float(np.quantile(same_product_scores, 0.90)),
    "random_cosine_mean": float(np.mean(random_pair_scores)),
    "random_cosine_median": float(np.median(random_pair_scores)),
}
alignment_metrics["median_margin"] = (
    alignment_metrics["same_cosine_median"] - alignment_metrics["random_cosine_median"]
)
alignment_metrics["passed"] = bool(
    alignment_metrics["same_cosine_median"] >= ALIGNMENT_MEDIAN_MIN
    and alignment_metrics["median_margin"] >= ALIGNMENT_MARGIN_MIN
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "alignment_report.json").write_text(
    json.dumps(alignment_metrics, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
pd.DataFrame({
    "same_product_cosine": pd.Series(same_product_scores),
    "random_pair_cosine": pd.Series(random_pair_scores),
}).to_parquet(OUTPUT_ROOT / "alignment_scores.parquet", index=False)

print(json.dumps(alignment_metrics, indent=2))
if STOP_IF_ALIGNMENT_FAILS and not alignment_metrics["passed"]:
    raise RuntimeError(
        "Alignment gate failed. Review the report before full encoding; "
        "set STOP_IF_ALIGNMENT_FAILS=False only if you intentionally accept this result."
    )

## 6. Encode toàn bộ metadata versions

Toàn bộ `normalized_text` được encode trong một lần gọi. `BATCH_SIZE` chỉ điều khiển mini-batch trên GPU; output được lưu thành một file NumPy duy nhất và có thể tái sử dụng khi chạy lại.

In [ ]:
version_embedding_path = OUTPUT_ROOT / "version_embeddings.f16.npy"
encoding_config_path = OUTPUT_ROOT / "version_encoding_config.json"
encoding_config = {
    "source_preprocessed_root": str(PREPROCESSED_ROOT),
    "model": MODEL_NAME,
    "embedding_dim": EMBEDDING_DIM,
    "max_sequence_length": max_sequence_length,
    "metadata_versions": len(product_versions),
}

reuse = False
if version_embedding_path.is_file():
    if not RESUME_VERSION_ENCODING:
        raise FileExistsError(
            "Version embeddings already exist. Enable RESUME_VERSION_ENCODING or RESET_OUTPUT."
        )
    if not encoding_config_path.is_file():
        raise FileNotFoundError("version_encoding_config.json is missing; safe resume is not possible.")
    previous_config = json.loads(encoding_config_path.read_text(encoding="utf-8"))
    if previous_config != encoding_config:
        raise ValueError(
            f"Encoding configuration changed; set RESET_OUTPUT=True to restart.\n"
            f"old={previous_config}\nnew={encoding_config}"
        )
    existing = np.load(version_embedding_path, mmap_mode="r")
    reuse = existing.shape == (len(product_versions), EMBEDDING_DIM) and existing.dtype == np.float16
    del existing

devices = [f"cuda:{index}" for index in range(torch.cuda.device_count())]
use_multi_gpu = USE_ALL_GPUS and len(devices) > 1
pool = model.start_multi_process_pool(target_devices=devices) if use_multi_gpu else None
print("encoding devices:", devices if use_multi_gpu else ["cuda:0"])

if not reuse:
    started = time.time()
    version_embeddings = encode_texts(
        product_versions["normalized_text"].tolist(),
        pool=pool,
    )
    temporary_path = version_embedding_path.with_suffix(".tmp")
    with temporary_path.open("wb") as handle:
        np.save(handle, version_embeddings.astype(np.float16))
    temporary_path.replace(version_embedding_path)
    encoding_config_path.write_text(
        json.dumps(encoding_config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"encoded {len(product_versions):,} rows in {time.time() - started:,.1f} seconds")
else:
    print("Reusing the existing version embedding file.")

if pool is not None:
    model.stop_multi_process_pool(pool)

expected_version_rows = int(global_products["version_count"].sum())
assert len(product_versions) == expected_version_rows
print("Version embeddings are ready.")

## 7. Aggregate normalized mean theo global product_id

Version embeddings đã normalized riêng. Bước này tính mean bằng float32, sau đó normalize lại và lưu global matrix bằng float16.

In [ ]:
num_products = len(global_products)
product_id_index = pd.Index(global_products["product_id"])
indices = product_id_index.get_indexer(product_versions["product_id"])
if (indices < 0).any():
    raise ValueError("product_versions contains a product_id that is absent from global_products")

version_embeddings = np.load(version_embedding_path, mmap_mode="r")
if version_embeddings.shape != (len(product_versions), EMBEDDING_DIM):
    raise ValueError(f"Unexpected version embedding shape: {version_embeddings.shape}")

sums = np.zeros((num_products, EMBEDDING_DIM), dtype=np.float32)
counts = np.zeros(num_products, dtype=np.int32)
np.add.at(sums, indices, np.asarray(version_embeddings, dtype=np.float32))
np.add.at(counts, indices, 1)

expected_counts = global_products["version_count"].to_numpy(dtype=np.int32)
assert np.array_equal(counts, expected_counts)

sums /= counts[:, None]
norms = np.linalg.norm(sums, axis=1, keepdims=True)
if (norms <= 0).any() or not np.isfinite(norms).all():
    raise ValueError("Invalid global mean embedding norms")
sums /= norms
global_embeddings = sums.astype(np.float16)

global_embedding_path = OUTPUT_ROOT / "global_product_embeddings.f16.npy"
temporary_path = global_embedding_path.with_suffix(".tmp")
with temporary_path.open("wb") as handle:
    np.save(handle, global_embeddings)
temporary_path.replace(global_embedding_path)

global_products.to_parquet(
    OUTPUT_ROOT / "global_embedding_index.parquet",
    index=False,
    compression="zstd",
)
print("Global embedding matrix written:", global_embedding_path)

## 8. Final validation và manifest

In [ ]:
global_embeddings_check = np.load(global_embedding_path, mmap_mode="r")
assert global_embeddings_check.shape == (num_products, EMBEDDING_DIM)
assert global_embeddings_check.dtype == np.float16

rng = np.random.default_rng(SEED)
check_indices = rng.choice(num_products, size=min(20_000, num_products), replace=False)
check_block = np.asarray(global_embeddings_check[check_indices], dtype=np.float32)
assert np.isfinite(check_block).all()
check_norms = np.linalg.norm(check_block, axis=1)
if not np.allclose(check_norms, 1.0, atol=2e-3):
    raise ValueError(f"Global embedding norms outside tolerance: {check_norms.min()}..{check_norms.max()}")

embedding_manifest = {
    "contract_version": "phase2-global-embeddings-v1",
    "source_preprocessed_root": str(PREPROCESSED_ROOT),
    "model": MODEL_NAME,
    "model_license": "CC-BY-NC-4.0",
    "base_embedding_dimension": int(base_dimension),
    "output_embedding_dimension": EMBEDDING_DIM,
    "max_sequence_length": max_sequence_length,
    "dtype": "float16",
    "global_products": num_products,
    "metadata_versions": expected_version_rows,
    "aggregation": "L2Normalize(Mean(L2NormalizedVersionEmbeddings))",
    "alignment": alignment_metrics,
    "files": {
        "embeddings": str(global_embedding_path),
        "index": str(OUTPUT_ROOT / "global_embedding_index.parquet"),
        "alignment_report": str(OUTPUT_ROOT / "alignment_report.json"),
    },
    "sample_norm": {
        "min": float(check_norms.min()),
        "mean": float(check_norms.mean()),
        "max": float(check_norms.max()),
    },
    "software": {
        "torch": torch.__version__,
        "sentence_transformers": installed_version("sentence-transformers"),
        "transformers": installed_version("transformers"),
    },
}
(OUTPUT_ROOT / "embedding_manifest.json").write_text(
    json.dumps(embedding_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(embedding_manifest, ensure_ascii=False, indent=2))
print("\nFinal embedding checks: PASSED")

## 9. Bàn giao cho RQ-VAE

In [ ]:
del global_embeddings_check
del global_embeddings
del version_embeddings
del sums
del counts

output_size = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob("*") if path.is_file())
print(f"Output size: {output_size / 2**30:,.2f} GiB")
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("\nNotebook 02 is complete when:")
print("- alignment passed = True")
print("- Final embedding checks: PASSED")
print("- global_product_embeddings.f16.npy and global_embedding_index.parquet have the same row count")
print("\nNext RQ-VAE inputs:")
print(OUTPUT_ROOT / "global_product_embeddings.f16.npy")
print(OUTPUT_ROOT / "global_embedding_index.parquet")